# Closest Polygon
Script to identify closest WDPA polygon to each project location

## Input Files, loaded into ArcGIS Pro
### Project locations (point)
Requires Year field giving year loan signed
- `Geolocated Assets/project_finance_loan_pairs`

### Protected Area Locations
Accessed from https://www.protectedplanet.net/en/data/wdpca and split into three different layers based on the Realm field (terrestrial/coastal/marine), retaining only polygons
- `Protected Areas/WDPA_marine`
- `Protected Areas/WDPA_terrestrial`
- `Protected Areas/WDPA_coastal`

In [2]:
import pdb
import datetime
import numpy as np
import arcpy

from sys import argv

def getYears(layer, yearName):
    return np.unique([row[0] for row in arcpy.da.SearchCursor(layer, yearName)])

def main(points, polygons, workspace, out_file):
    point_years = getYears(points, 'Year')
    poly_years = getYears(polygons, 'STATUS_YR')
    poly_year_min = np.min(poly_years)
    joined_list = []
    for year in point_years:
        ## Select point in year
        points_year = "points%0d" % year
        arcpy.management.MakeFeatureLayer(
            points,
            points_year,
            "Year=%0d" % year
        )
        ## Selected polygons in or before year
        poly_valid = "polygonsValid%0d" % year
        arcpy.management.MakeFeatureLayer(
            polygons,
            poly_valid,
            "STATUS_YR<=%0d" % year
        )
        ## Find closest polygon(s) to each unintersected point
        joined = "joined%0d" % year
        arcpy.analysis.SpatialJoin(
            points_year,
            poly_valid,
            joined,
            join_operation="JOIN_ONE_TO_MANY",
            join_type="KEEP_ALL",
            match_option="CLOSEST_GEODESIC",
            distance_field_name="polygonDistance",
        )
        ## Delete temporary layesr
        arcpy.management.Delete(points_year)            
        arcpy.management.Delete(poly_valid)
        ## Add joined file to list
        joined_list += [joined]
    ## Combine joined list into single output
    arcpy.management.Merge(joined_list,
                           out_file)
    ## Delete temporary layers
    for joined in joined_list:
        arcpy.management.Delete(joined)

In [ ]:
point_files = {
    'loans': 'Geolocated Assets/project_finance_loan_pairs',
}
poly_files = {
    'wdpa_coastal': 'Protected Areas/WDPA_coastal',
    'wdpa_marine': 'Protected Areas/WDPA_marine',
    'wdpa_terrestrial': 'Protected Areas/WDPA_terrestrial'
}

if __name__ == '__main__':
    # Global Environment settings
    workspace = '.'
    with arcpy.EnvManager(scratchWorkspace=workspace, workspace=workspace):
        print(datetime.datetime.now())
        for point_key in list(point_files.keys()):
            for poly_key in poly_files.keys():
                out_file = '_'.join([point_key, poly_key])
                print(out_file)
                main(point_files[point_key],
                     poly_files[poly_key],
                     workspace,
                     out_file)